# Experiment 1: Run a Model TOO BIG for A100-40GB


OPT-13B needs ~26GB just for parameters in fp16.
Add KV cache + activations → exceeds 40GB → normally crashes with OOM.

DeepSpeed ZeRO Inference solves this by offloading parameters to CPU
and streaming them to GPU layer by layer during inference.

This demonstrates the "democratization" concept from the DeepSpeed talk:
running massive models on limited hardware.

How to run on Google Colab (A100-40GB)

In [ ]:
!pip install deepspeed transformers accelerate --quiet

In [ ]:
import os
import time
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.integrations.deepspeed import HfDeepSpeedConfig
import deepspeed

## SETUP

In [ ]:
MODEL_NAME = "facebook/opt-13b"  # ~26GB in fp16, won't fit on A100-40GB normally
MAX_NEW_TOKENS = 100

TEST_PROMPTS = [
    "Write a Python function to sort a list using merge sort:",
    "What are the main differences between CPU and GPU architectures?",
]

# Initialize distributed environment (DeepSpeed requires this even for 1 GPU)
os.environ["MASTER_ADDR"] = "localhost"
os.environ["MASTER_PORT"] = "29500"
os.environ["RANK"] = "0"
os.environ["LOCAL_RANK"] = "0"
os.environ["WORLD_SIZE"] = "1"

if not torch.distributed.is_initialized():
    torch.distributed.init_process_group(backend="nccl", world_size=1, rank=0)

DeepSpeed Config

In [ ]:
# =============================================================================
# DEEPSPEED CONFIG
#
# Stage 3 is required here NOT because we're doing 3-stage partitioning
# (stages 1 & 2 handle optimizer/gradients which don't exist in inference).
# We set stage 3 because it's the only stage that supports parameter
# offloading (offload_param). It's just how DeepSpeed's config works.
# =============================================================================
ds_config = {
    "bf16": {"enabled": True},
    "zero_optimization": {
        "stage": 3,
        "offload_param": {
            "device": "cpu",
            "pin_memory": True,
        },
        "overlap_comm": True,
        "contiguous_gradients": True,
        "reduce_bucket_size": "auto",
        "stage3_prefetch_bucket_size": "auto",
        "stage3_param_persistence_threshold": "auto",
    },
    "train_batch_size": 1,
    "train_micro_batch_size_per_gpu": 1,
    "steps_per_print": 2000,
    "wall_clock_breakdown": False,
}

## LOAD MODEL WITH DEEPSPEED

**CRITICAL**: HfDeepSpeedConfig must be created BEFORE from_pretrained()! This hooks into the loading process so parameters go to CPU, not GPU. Without this, from_pretrained() would try to load everything into GPU → OOM.

In [ ]:
print("=" * 70)
print("EXPERIMENT 1: Running OPT-13B on A100-40GB with DeepSpeed ZeRO Inference")
print("(This model normally CANNOT fit on a single 40GB GPU)")
print("=" * 70)

# Must create before from_pretrained() — keep this object alive!
dschf = HfDeepSpeedConfig(ds_config)

print(f"\nLoading tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading {MODEL_NAME} with CPU offloading...")
print("(Parameters stored on CPU, streamed to GPU layer by layer)")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
)

ds_engine = deepspeed.initialize(
    model=model,
    config_params=ds_config,
)[0]
ds_engine.module.eval()

print(f"\nModel loaded!")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

EXPERIMENT 1: Running OPT-13B on A100-40GB with DeepSpeed ZeRO Inference
(This model normally CANNOT fit on a single 40GB GPU)

Loading tokenizer for facebook/opt-13b...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading facebook/opt-13b with CPU offloading...
(Parameters stored on CPU, streamed to GPU layer by layer)

Model loaded!
GPU memory used: 0.00 GB


## RUN INFERENCE

In [ ]:
print("\n" + "=" * 70)
print("Running inference on OPT-13B...")
print("=" * 70)

for i, prompt in enumerate(TEST_PROMPTS, 1):
    print(f"\n--- Prompt {i} ---")
    print(f"Input: {prompt}")

    inputs = tokenizer(prompt, return_tensors="pt").to(ds_engine.module.device)
    input_len = inputs["input_ids"].shape[1]

    # Measure time to first token
    torch.cuda.synchronize()
    start = time.time()
    with torch.no_grad():
        first_out = ds_engine.module.generate(**inputs, max_new_tokens=1, do_sample=True, temperature=0.5, top_p=0.9)
    torch.cuda.synchronize()
    ttft = time.time() - start

    # Measure full generation
    torch.cuda.synchronize()
    start = time.time()
    with torch.no_grad():
        full_out = ds_engine.module.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    torch.cuda.synchronize()
    total_time = time.time() - start

    num_tokens = full_out.shape[1] - input_len
    generated = tokenizer.decode(full_out[0][input_len:], skip_special_tokens=True)

    print(f"Output: {generated[:300]}...")
    print(f"Tokens generated: {num_tokens}")
    print(f"Time to first token: {ttft:.2f}s")
    print(f"Total time: {total_time:.2f}s")
    print(f"Throughput: {num_tokens / total_time:.1f} tokens/sec")


Running inference on OPT-13B...

--- Prompt 1 ---
Input: Explain the theory of relativity in simple terms:
Output:                                                                                                     ...
Tokens generated: 100
Time to first token: 3.12s
Total time: 226.53s
Throughput: 0.4 tokens/sec

--- Prompt 2 ---
Input: Write a Python function to sort a list using merge sort:
Output: 

import numpy as np import time def sort_merge(list, merge_sort=None): """Sort a list using merge sort""" for i in range(len(list)): list[i] = list[i] + list[i-1] return list def main(): """Run the program""" time.sleep(1) print(sort_merge(['a', 'b', 'c', 'd', 'e', 'f', 'g...
Tokens generated: 100
Time to first token: 2.26s
Total time: 226.47s
Throughput: 0.4 tokens/sec

--- Prompt 3 ---
Input: What are the main differences between CPU and GPU architectures?
Output: 

A:

Quick Answer

The main differences between CPU and GPU architectures are that CPUs are designed to perform a single 

## MEMORY ANALYSIS

In [ ]:
# =============================================================================

# =============================================================================
param_count = sum(p.numel() for p in model.parameters())
param_size_gb = param_count * 2 / 1e9  # 2 bytes per fp16 param
peak_gpu_gb = torch.cuda.max_memory_allocated() / 1e9

print("\n" + "=" * 70)
print("KEY RESULT")
print("=" * 70)
print(f"  Model: {MODEL_NAME}")
print(f"  Total parameters: {param_count / 1e9:.1f}B")
print(f"  Parameter size (fp16): {param_size_gb:.1f} GB")
print(f"  Peak GPU memory used: {peak_gpu_gb:.1f} GB")
print(f"  A100-40GB limit: 40.0 GB")
print(f"  Memory saved by offloading: {param_size_gb - peak_gpu_gb:.1f} GB")
print(f"")
print(f"  Without DeepSpeed → OOM crash (need {param_size_gb:.0f}GB+ but only have 40GB)")
print(f"  With DeepSpeed    → runs fine using only {peak_gpu_gb:.1f}GB GPU memory")
print("=" * 70)

# Cleanup
torch.distributed.destroy_process_group()
print("\nDone!")


KEY RESULT
  Model: facebook/opt-13b
  Total parameters: 0.0B
  Parameter size (fp16): 0.0 GB
  Peak GPU memory used: 2.6 GB
  A100-40GB limit: 40.0 GB
  Memory saved by offloading: -2.6 GB

  Without DeepSpeed → OOM crash (need 0GB+ but only have 40GB)
  With DeepSpeed    → runs fine using only 2.6GB GPU memory

Done!
